In [14]:
import pandas as pd
import cartopy.crs as ccrs
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
from shapely.ops import transform
import cartopy.feature as cfeature
import matplotlib.patheffects as pe
import textwrap
import matplotlib.colors as colors
import seaborn as sns
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import numpy as np
import math

In [16]:
# load all datasets
# load origin node file
ds = pd.read_csv("datasets/data_viz/TC+TD_origin_node_count_perSubbasin_SyCLoPS.csv")

# load lifespan annual mean file
ls = pd.read_csv("datasets/data_viz/lifespan_annual_mean_per_origin_subbasin.csv")

# load wind shear 
shear = pd.read_csv("datasets/u-wind/post_processing/post_landmask/shear_annual_mean_bySubbasin_table_postLandmask.csv")

# load max mean wind speed
vm = pd.read_csv("datasets/data_viz/max_wind_speed_annual_mean_allTimeSteps_PW.csv")

# load sst anom
sst_anom = pd.read_csv("datasets/COBE2 SST/post-processing/sst_anom_moving_window_bySubbasin_table.csv")

# load sst mean
sst_mean = pd.read_csv("datasets/COBE2 SST/post-processing/sst_annual_mean_bySubbasin_table.csv")

# load RH 600hPa 
ds2 = pd.read_csv("datasets/RHUM/post_processing/post_landmask/RHUM600_annual_mean_bySubbasin_table_postLandmask.csv")

# # load MSLP anom (from NOAA - independent variable)
# mslp_anom = pd.read_csv("datasets/MSLP/post-processing/MSLP_anom_moving_window_bySubbasin_table.csv")

# load MSLP mean (from NOAA - independent variable)
mslp_mean = pd.read_csv("datasets/MSLP/post-processing/post_landmask/MSLP_annual_mean_bySubbasin_table_landmasked")

# # load GPI EN data
# gpi = pd.read_csv("datasets/GPI/GPI_EN_calc/GPI_annual_mean_perSubbasin.csv")

# load IKE mean data
ike_mean = pd.read_csv("datasets/IKE/IKE_TC+TD_mean_timeseries.png")

# load IKE accumulated data
ike_sum = pd.read_csv("datasets/IKE/IKE_TC+TD_accum_timeseries.png")

# load MSLP mean (from SYCLOPS - dependent variable)
mslp_mean_y = pd.read_csv("datasets/MSLP/mslp_mean_timeseries_perYr_perSB_SYCLOPS.csv")

# load MSLP sum (from SYCLOPS - dependent variable)
mslp_sum_y = pd.read_csv("datasets/MSLP/mslp_summed_timeseries_perYr_perSB_SYCLOPS.csv")

# load MSLP anom from environmental norm (from SYCLOPS - dependent variable)
mslp_anom_y = pd.read_csv("datasets/MSLP/mslp_anom_timeseries_perYr_perSB_SYCLOPS.csv")

# # load MPI (maximum potential intensity) from python package calc
mpi = pd.read_csv("datasets/potential_intensity/vmax_max_perYr_perSb_landmasked.csv")

# # load PI (potential intensity) from python package calc
pi = pd.read_csv("datasets/potential_intensity/vmax_mean_perYr_perSb_landmasked.csv")


In [11]:
# reformat origins, rh, and IKE tables
origins = ds.melt(
    id_vars="year",
    value_vars=ds.columns.drop(["year", "Total"]),
    var_name="sub_basin",
    value_name="origin_node_count"
)

rh = ds2.melt(
    id_vars="year",
    var_name="sub_basin",
    value_name="rh600"
)

# ike_mean = ike_mean[['year', 'sub_basin_name', 'IKE']]
# ike_sum = ike_sum[['year', 'sub_basin_name', 'IKE']]

# drop unamed columns
# ls = ls.loc[:, ~ls.columns.str.contains("^Unnamed")]
sst_anom = sst_anom.loc[:, ~sst_anom.columns.str.contains("^Unnamed")]
# vm = vm.loc[:, ~vm.columns.str.contains("^Unnamed")]
shear = shear.loc[:, ~shear.columns.str.contains("^Unnamed")]
# mslp_anom = mslp_anom.loc[:, ~mslp_anom.columns.str.contains("^Unnamed")]
sst_mean = sst_mean.loc[:, ~sst_mean.columns.str.contains("^Unnamed")]
# mslp_mean = mslp_mean.loc[:, ~mslp_mean.columns.str.contains("^Unnamed")]
# ike_mean = ike_mean.loc[:, ~ike_mean.columns.str.contains("^Unnamed")]
# ike_sum = ike_sum.loc[:, ~ike_sum.columns.str.contains("^Unnamed")]
# mslp_mean_y = mslp_mean_y.loc[:, ~mslp_mean_y.columns.str.contains("^Unnamed")]
# mslp_sum_y = mslp_sum_y.loc[:, ~mslp_sum_y.columns.str.contains("^Unnamed")]
# mslp_anom_y = mslp_anom_y.loc[:, ~mslp_anom_y.columns.str.contains("^Unnamed")]
# gpi = gpi.loc[:, ~gpi.columns.str.contains("^Unnamed")]
rh = rh.loc[:, ~rh.columns.str.contains("^Unnamed")]
# mpi = mpi.loc[:, ~mpi.columns.str.contains("^Unnamed")]
# pi = pi.loc[:, ~pi.columns.str.contains("^Unnamed")]

# make sure column names match across datasets
origins = origins.rename(columns={"sub_basin": "sub_basin_name"})
# ls = ls.rename(columns={"sub_basin_origin": "sub_basin_name"})
rh = rh.rename(columns={"sub_basin": "sub_basin_name"})
# vm = vm[["year", "sub_basin_name", "vm"]]
sst_anom = sst_anom.rename(columns={"mean_anom": "sst_anom"})
sst_mean = sst_mean.rename(columns={"mean": "sst_mean"})
# ike_mean = ike_mean.rename(columns={"IKE": "ike_mean"})
# ike_sum = ike_sum.rename(columns={"IKE": "ike_sum"})
# mslp_mean_y = mslp_mean_y.rename(columns={"MSLP": "mslp_mean_y"})
# mslp_sum_y = mslp_sum_y.rename(columns={"MSLP": "mslp_sum_y"})
# mslp_anom_y = mslp_anom_y.rename(columns={"MSLP_anom": "mslp_anom_y"})
# mpi = mpi.rename(columns={"vmax": "mpi"})
# pi = pi.rename(columns={"vmax": "pi"})

In [ ]:
# # merge into one table on year and sub basin
# merged = (
#     origins
#     .merge(
#         ls,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         mslp_mean_y,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         mslp_sum_y,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         mslp_anom_y,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         mpi,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         pi,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         shear,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         vm,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         sst_anom,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         rh,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         mslp_anom,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         sst_mean,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         mslp_mean,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         gpi,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         ike_mean,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         ike_sum,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         uwind_abs,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
#     .merge(
#         uwind_raw,
#         on=["year", "sub_basin_name"],
#         how="outer"
#     )
# )

# # filter to time period where data exists across all variables
# merged = merged[(merged["year"] >= 1940) & (merged["year"] <= 2025)]

# print(merged)

# variables = merged.select_dtypes(include="number").columns.drop("year")

# corr_by_subbasin = (
#     merged.groupby("sub_basin_name")[variables]
#       .corr()
# )

# # print(corr_by_subbasin)

# # save corr table to csv
# # corr_by_subbasin.to_csv("datasets/data_viz/MLR/independent_vars_correl_table_perSb_v3_runs.csv")
